In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from typing import Dict,Any
import os

from pydantic.v1.fields import DeferredType


load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_API_URL_RESPONSE")

system_prompt = """
你是遵循 ReAct 范式的智能助手。
可用工具：

{tools}
math(number1, number2, operator)：二元四则运算，operator只能是 + - * /。
⚠️ 注意：math工具只能一次计算两个数字！复杂算式需要分步多次调用。

输出严格只能两种格式，禁止额外文字：
【需要调用工具】
Thought: 推理思考
Action: math(数字1,数字2,运算符)

【得到最终答案】
Thought: 信息充足，可以汇总答案
Finish: 最终回答

示例分步计算：
Question: (10+20)*2等于多少？
Thought: 先计算10+20
Action: math(10,20,+)

Observation: 30
Thought: 再用上一步结果30乘以2
Action: math(30,2,*)

Observation: 60
Thought: 计算完成
Finish: (10+20)*2 的结果是60

Observation 由系统提供，不要自行生成。
"""




def math_tool(number1:float, number2:float, operator:str) -> float:
    """ 
    计算数学表达式
    :param number1: 第一个数字
    :param number2: 第二个数字
    :param operator: 运算符
    :return: 计算结果
    """
    if operator == "+":
        return number1 + number2
    elif operator == "-":
        return number1 - number2
    elif operator == "*":
        return number1 * number2
    elif operator == "/":
        if number2 == 0:
            raise ValueError("除数不能为0")
        return number1 / number2
    else:
        raise ValueError(f"无效的运算符: {operator}")


class ToolExecutor:
    def __init__(self):
        self.tools :Dict[str,Dict[str,Any]] = {}
    def register_tool(self, name:str, description:str, function:callable):
        """ 注册工具
        """
        if name in self.tools:
            print(f"工具 {name} 已注册，将覆盖")
        self.tools[name] = {"description": description, "function": function}
        print(f"工具 {name} 已注册")
    def getTools(self, name:str) -> callable:
        """ 获取工具函数
        """
        return self.tools.get(name, {}).get("function")
    def getAvailableTools(self) -> str:
        """ 
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
                f"- {name}: {info['description']}" 
                for name, info in self.tools.items()
        ])

# --- 工具初始化与使用示例 ---
if __name__ == '__main__':
    # 1. 初始化工具执行器
    toolExecutor = ToolExecutor()

    # 2. 注册我们的工具
    math_description = "一个计算工具箱，可以计算数学表达式。"
    toolExecutor.register_tool("math", math_description, math_tool)
    
    # 3. 打印可用的工具
    print("\n--- 可用的工具 ---")
    print(toolExecutor.getAvailableTools())

    # 4. 智能体的Action调用，这次我们问一个实时性的问题
    print("\n--- 执行 Action: math['(8*6)-(8/2)+3'] ---")
    tool_name = "math"
    tool_input = "8*6-(8/2)+3"

    tool_function = toolExecutor.getTools(tool_name)

class ReactAgent:
    def __init__(self, api_key: str,base_url: str, system_prompt: str, tool_executor: ToolExecutor, max_steps: int = 5):
        self.api_key = api_key
        self.base_url = base_url
        self.model = "qwen3.8-max"
        self.max_steps = max_steps
        self.toolExecutor = tool_executor
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        self.system_prompt = system_prompt
    

    def llmcall(self, content:str) -> str:
        """ 调用LLM
        """


工具 math 已注册

--- 可用的工具 ---
- math: 一个计算工具箱，可以计算数学表达式。

--- 执行 Action: math['(8*6)-(8/2)+3'] ---


In [ ]:
import json
from openai import OpenAI
from typing import Dict, Any, Optional
from datetime import datetime
import os

from dotenv import load_dotenv

load_dotenv()

# 从环境提取 API 密钥和基础 URL
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_API_URL")

# ==============================
# 二元数学工具（你原有代码保持不变）
# ==============================
def math_tool(number1: float, number2: float, operator: str) -> float:
    """
    二元数学运算工具
    :param number1: 第一个数字
    :param number2: 第二个数字
    :param operator: 运算符，支持 + - * /
    :return: 计算结果
    """
    if operator == "+":
        return number1 + number2
    elif operator == "-":
        return number1 - number2
    elif operator == "*":
        return number1 * number2
    elif operator == "/":
        if number2 == 0:
            raise ValueError("除数不能为0")
        return number1 / number2
    else:
        raise ValueError(f"无效运算符: {operator}，仅支持 + - * /")

def get_time() -> str:
    """ 获取当前时间 """
    return datetime.now().hour
# ==============================
# 工具执行器（你原有代码，仅保留）
# ==============================
class ToolExecutor:
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def register_tool(self, name: str, description: str, function: callable):
        """注册工具"""
        if name in self.tools:
            print(f"工具 {name} 已注册，将覆盖")
        self.tools[name] = {"description": description, "function": function}
        print(f"工具 {name} 已注册")

    def getTools(self, name: str) -> Optional[callable]:
        """获取工具函数"""
        return self.tools.get(name, {}).get("function")

    def getAvailableTools(self) -> str:
        """获取所有可用工具描述字符串"""
        return "\n".join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])

    def execute_tool(self, tool_name: str, args: dict) -> str:
        """执行工具，捕获异常，返回Observation字符串"""
        func = self.getTools(tool_name)
        if not func:
            return f"[错误] 不存在工具：{tool_name}"
        try:
            res = func(**args)
            return str(res)
        except Exception as e:
            return f"[工具执行异常] {str(e)}"


# ==============================
# OpenAI SDK 兼容版 JSON ReAct Agent
# ==============================
class JsonReActAgent:
    def __init__(self, api_key: str,base_url: str, tool_executor: ToolExecutor, max_iter: int = 8):
        # 初始化OpenAI客户端，对接DashScope兼容接口
        self.client = OpenAI(
            api_key=api_key,
            base_url=base_url
        )
        self.model_name = "qwen3.8-max"
        self.max_iter = max_iter
        self.tool_executor = tool_executor

        self.system_prompt = f"""
你是遵循 ReAct 范式的智能助手。
可用工具：
{self.tool_executor.getAvailableTools()}
math(number1, number2, operator)：二元四则运算，operator只能是 + - * /。
⚠️ math工具一次仅支持两个数字运算，复杂算式需要分步多次调用。

## 硬性规则
你的输出**只能返回一段纯净JSON**，禁止输出任何额外文字、注释、Markdown、解释！
不要使用```json代码块标记，直接输出JSON文本。

两种JSON格式二选一：

### 1. 需要调用工具(type="action")
{{
  "thought": "你的推理思考过程",
  "type": "action",
  "tool": "工具名称",
  "args": {{
    "number1": 数值,
    "number2": 数值,
    "operator": "运算符"
  }}
}}

### 2. 推理完成，输出答案(type="finish")
{{
  "thought": "信息充足，可以汇总最终答案",
  "type": "finish",
  "answer": "最终回答内容"
}}

Observation 是系统执行工具返回的结果，由系统提供，不要自己生成。
"""

    def llm_call(self, content: str) -> str:
        """使用OpenAI SDK调用大模型"""
        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": content}
            ],
            temperature=0,
        )
        raw_text = response.choices[0].message.content.strip()
        # 清洗模型偶尔输出的代码块标记
        raw_text = raw_text.removeprefix("```json").removesuffix("```").strip()
        return raw_text

    def parse_json_output(self, text: str) -> Optional[dict]:
        """尝试解析JSON，失败返回None"""
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            print(f"❌ JSON解析失败，模型原始输出：\n{text}\n")
            return None

    def run(self, question: str) -> str:
        trajectory = f"Question: {question}\n"
        print(f"==== 用户问题 ====\n{question}\n")

        for step in range(self.max_iter):
            print(f"-------- 迭代 {step+1} --------")
            llm_raw = self.llm_call(trajectory)
            print(f"模型原始输出:\n{llm_raw}\n")

            data = self.parse_json_output(llm_raw)
            if data is None:
                # JSON非法，追加警告继续循环
                warn_msg = "\n【警告】输出不是合法JSON！严格按照要求只输出纯净JSON\n"
                trajectory += llm_raw + warn_msg
                continue

            trajectory += json.dumps(data, ensure_ascii=False, indent=2) + "\n"

            if data["type"] == "finish":
                ans = data["answer"]
                print(f"✅ 推理结束！答案：{ans}")
                return ans

            elif data["type"] == "action":
                tool_name = data["tool"]
                args = data["args"]
                obs_result = self.tool_executor.execute_tool(tool_name, args)
                print(f"Observation: {obs_result}\n")
                trajectory += f"Observation: {obs_result}\n"

        return f"⚠️ 达到最大迭代次数{self.max_iter}，推理终止"


# ==============================
# 程序入口
# ==============================
if __name__ == '__main__':
    # 注册工具
    toolExecutor = ToolExecutor()
    math_desc = "二元四则运算工具，输入两个数字与运算符（+ - * /）进行计算"
    toolExecutor.register_tool("math", math_desc, math_tool)
    toolExecutor.register_tool("get_time", "获取当前时刻的小时数字", get_time)

    # 填入你的 DashScope sk-xxx

    agent = JsonReActAgent(
        api_key=api_key,
        base_url=base_url,
        tool_executor=toolExecutor,
        max_iter=8
    )

    # 测试问题，会自动分步运算
    agent.run("计算当前时间的小时数字*2-3")